# Applying the Robust Multi-Neuron GLM-HMM Framework to an Example Medial Prefrontal Cortex Dataset

This notebook demonstrates how to apply the robust multi-neuron GLM-HMM framework to an example medial prefrontal cortex (mPFC) dataset including  the creation of the predictor matrix and visualization of the results.

```matlab
clc;
clear all;
% Load the data
behav=readmatrix("150628_Behavior.dat");
cells=readmatrix("150628_CellType.dat");
pos=readmatrix("150628_Pos.dat");
spikes=readmatrix("150628_SpikeData.dat");
num_cells=size(cells,1);
```

```matlab
% Extract neurons
for i=1:num_cells
    temp=find(spikes(:,2)==i);
    RAW.Nrast{i,1}=spikes(temp,1)/1000; %in seconds
    clear temp;
end

% Bin neuronal data across the entire session
N=[];
bin_size=0.05;
FR=zeros(1,num_cells);
for neuron_id=1:num_cells
    temp_binned_spikes=[];
    for i=1:size(behav,1)-1

        Tanalysis=behav(i,1)/1000 :bin_size: behav(i,2)/1000+(behav(i+1,1)-behav(i,2))/1000;
        temp_binned_spikes=[temp_binned_spikes,histcounts(RAW.Nrast{neuron_id,1}(find(RAW.Nrast{neuron_id,1}>=Tanalysis(1) & RAW.Nrast{neuron_id,1}<=Tanalysis(end))),[Tanalysis Tanalysis(end)+(Tanalysis(end)-Tanalysis(end-1))])]; %spike counts
      
     end
    N.binned_spikes(:,neuron_id)=temp_binned_spikes';
    RAW.FR(neuron_id)=sum(N.binned_spikes(:,neuron_id))/(size(N.binned_spikes,1)*bin_size);
    RAW.neuron_sum(neuron_id)=sum(N.binned_spikes(:,neuron_id));
end

% Bin predictors across the entire session

N.binned_trialstart=[];
N.binned_trialend=[];
N.binned_outcome=[];
N.binned_trialno=[];
trial_start=behav(:,1)/1000;
trial_end=behav(:,2)/1000;
for i=1:size(behav,1)-1
    temp_binned_trialstart=[];
    Tanalysis=behav(i,1)/1000 :bin_size: behav(i,2)/1000+(behav(i+1,1)-behav(i,2))/1000;
    temp_binned_trialstart=histcounts(trial_start(find(trial_start>=Tanalysis(1) & trial_start<=Tanalysis(end))),[Tanalysis Tanalysis(end)+(Tanalysis(end)-Tanalysis(end-1))]);
    temp_binned_trialend=histcounts(trial_end(find(trial_end>=Tanalysis(1) & trial_end<=Tanalysis(end))),[Tanalysis Tanalysis(end)+(Tanalysis(end)-Tanalysis(end-1))]);
    N.binned_trialstart=[N.binned_trialstart;temp_binned_trialstart'];
    N.binned_trialend=[N.binned_trialend;temp_binned_trialend'];
    N.binned_trialno=[N.binned_trialno;i*ones(1,length(Tanalysis))'];
    if behav(i,4)==1 %correct
        N.binned_outcome=[N.binned_outcome;ones(1,length(Tanalysis))'];
    elseif behav(i,4)==0 %incorrect
        N.binned_outcome=[N.binned_outcome;-1*ones(1,length(Tanalysis))'];
    end
   
end
```

```matlab
% Create the basis functions
X=struct();
rcSelect = 2:6;
numRC = numel(rcSelect);
%bases = makeNonlinearRaisedCos(20, bin_size*1000, [0 1500], 5); % Pillow lab for 2.7s with 100ms 
if bin_size==0.05
    bases = makeNonlinearRaisedCos(20, bin_size*1000, [0 1500], 10); %for 50ms with 2.7s time support
elseif bin_size==0.02
    bases = makeNonlinearRaisedCos(20, bin_size*1000, [0 1000], 20); %for 20ms with 2.7s time support
end
figure;
plot(bases.tr * (bin_size), bases.B(:,rcSelect)); % convert bins back to seconds
xlabel('Time (s)');
ylabel('Basis amplitude');
title('Log raised cosine bases');
```

<img src="Figures\basis_mpfc.png" alt="basis" width="500">

```matlab
% Prepare the feature matrix
% After convolution, z-score each regressor column-wise

X=struct();
X.trialstart = zscore(temporalBases_sparse(N.binned_trialstart,bases.B(:,rcSelect)));
X.start_outcome = zscore(temporalBases_sparse(N.binned_outcome.*N.binned_trialstart, bases.B(:, rcSelect)));
X.pre_trialend=zscore(flipud(temporalBases_sparse(flipud(N.binned_trialend), bases.B(:, rcSelect))));
X.pre_end_outcome=zscore(flipud(temporalBases_sparse(flipud(N.binned_trialend.*N.binned_outcome), bases.B(:, rcSelect))));
X.trialend= zscore(temporalBases_sparse(N.binned_trialend,bases.B(:,rcSelect)));
X.end_outcome = zscore(temporalBases_sparse(N.binned_outcome.*N.binned_trialend, bases.B(:, rcSelect)));

for neuron_id=1:length(RAW.Nrast)
    y=N.binned_spikes(:, neuron_id);
    
    X.SpkHist{neuron_id}=zscore(temporalBases_sparse([0; y(1:end-1)], bases.B(:,rcSelect)));
end
```

```matlab
% Fit a sharded-state GLM-HMM and save results

ablation_type=0; %0 for glm with ridge, 1 for pure glm, 2 for random penalization
numStates=4;  % hard code
results = struct();
numNeurons = length(RAW.Nrast);
A0 = 1*eye(numStates)+.2*rand(numStates); %from Pillow lab
A0 = A0 ./ sum(A0,2);  % each row sums to 1
X_all_neuron = cell(1,numNeurons);
y_neuron = cell(1,numNeurons);
W0_neuron = cell(1,numNeurons);
optimal_lambda=nan(1,numNeurons);
for neuron_id=1:length(RAW.Nrast)
    y = N.binned_spikes(:,neuron_id);
    mean_y=mean(y);
    var_y=var(y);
    
    if RAW.FR(neuron_id)<0.2 
         disp(['Skipping low FR (<0.2Hz) neuron# ' num2str(neuron_id)]);
         continue;
    end
  
    % === 1. Prepare the feature matrix by combining all predictors ===
    X_all = [ ones(length(X.SpkHist{1,neuron_id}),1),  X.trialstart, X.start_outcome, X.pre_trialend, X.pre_end_outcome, X.trialend, X.end_outcome, X.SpkHist{1,neuron_id}];

    X_all_neuron{neuron_id}=X_all';

    % === 2. Initialize HMM params ===
    if ablation_type==0 %using glmnet with lambda
        opts = struct('alpha', 0, 'nfolds', 5,'nlambda',50,'standardize',false);
        cv_fit = cvglmnet(X_all(:,2:end), y, 'poisson', opts);
        optimal_lambda(neuron_id) = cv_fit.lambda_min;
        Wglm_ml = glmnetCoef(cv_fit.glmnet_fit, optimal_lambda(neuron_id));
        W0 = Wglm_ml + randn(size(X_all,2),numStates)*norm(Wglm_ml)*.25; %smart initialization

    elseif ablation_type==1 %using glmfit with no lambda
        [Wglm_ml, ~, ~] = glmfit(X_all(:,2:end), y, 'poisson', 'constant', 'on');
        optimal_lambda(neuron_id) = 0;
        W0 = Wglm_ml + randn(size(X_all,2),numStates)*norm(Wglm_ml)*.25; 

    elseif ablation_type==2 %not using any glm

        W0 = randn(size(X_all,2), numStates) * 0.01;
        W0(1,:) = log(mean(y) + 1e-6);
        optimal_lambda(neuron_id)=0;
    end
        
    W0_neuron{neuron_id}=W0;
    y_neuron{neuron_id}=y';
end

X_all_neuron = X_all_neuron(~cellfun(@isempty,X_all_neuron));
y_neuron = y_neuron(~cellfun(@isempty,y_neuron));
W0_neuron = W0_neuron(~cellfun(@isempty,W0_neuron));
optimal_lambda = optimal_lambda(~isnan(optimal_lambda));


%[Ahat, What, logp,logpTrace,sqErr,jj,dlogp,gams,all_weights_iter]=runEMforGLMHMM_multi_NR(A0,W0_neuron,X_all_neuron,y_neuron, optimal_lambda); %pure NR
[Ahat, What, logp,logpTrace,sqErr,jj,dlogp,gams,all_weights_iter]=runEMforGLMHMM_multi(A0,W0_neuron,X_all_neuron,y_neuron, optimal_lambda);

finalLogLikelihood=logpTrace(end);
logLs = finalLogLikelihood;

% --- Compute # of params ---
nCov = size(X_all,2);
p = numStates*(numStates-1) + numStates*nCov*length(y_neuron);  % transition + GLM weights

% --- Compute AIC/BIC ---
T = size(X_all,1);  % number of bins
AICs = -2*finalLogLikelihood + 2*p;
BICs = -2*finalLogLikelihood + p*log(T);

results.K = numStates;
results.Ahat = Ahat;
results.What = What;
results.gams = gams;
results.logpTrace = logpTrace;
results.finalLogL = logpTrace(end);
results.AIC = AICs;
results.BIC = BICs;
results.sqErr = sqErr;
results.nParams = p;
results.A0=A0;
results.optimal_lambda=optimal_lambda;
results.W0=W0_neuron;
results.all_weights_iter=all_weights_iter;
```

```matlab
% Check model convergence

fig = figure;
logpTrace=results.logpTrace(2:end);
plot(1:numel(logpTrace), -logpTrace, 'o-','LineWidth',2);
xlabel('Iteration');
ylabel('Negative log-likelihood');
box off

fig=figure;
sqErr=results.sqErr(2:end);
plot(1:numel(sqErr), sqErr, 'o-','LineWidth',2);
xlabel('Iteration');
ylabel('Squared parameter change');
box off
ylim([0 Inf])
```

<img src="Figures/logpTrace_mpfc.png" alt="logpTrace" width="600">
<img src="Figures/sqErr_mpfc.png" alt="sqerror" width="600">

```matlab
% Check correlation between states
numStates = results.K;
numNeurons = numel(results.What);
numPredictors = size(results.What{1,1},1);

% Stack all neuron weights into one array
corrMatAll = nan(numStates,numStates,numNeurons);

for n = 1:numNeurons
    corrMatAll(:,:,n) = corr(results.What{1,n});  % correlation across states
end

% Mean or median correlation across neurons
corrMat_mean = mean(corrMatAll,3);

fig=figure;
imagesc(corrMat_mean);
colormap(parula);
colorbar;
xticks(1:results.K)
xticklabels(['State 1';'State 2';'State 3';'State 4'])
yticks(1:results.K)
yticklabels(['State 1';'State 2';'State 3';'State 4'])
axis square;
title('Mean Correlation of GLM Weights (per neuron)');
```

<img src="Figures/statecorr_mpfc_4states.png" alt="Correlation Matrix" width="500">

```matlab
% Plotting gams
% find max probability states
[~, state_seq] = max(results.gams, [], 1);   % 1 x T
correct_start_times=find(N.binned_outcome.*N.binned_trialstart==1);
correct_end_times=find(N.binned_outcome.*N.binned_trialend==1);
incorrect_start_times=find(N.binned_outcome.*N.binned_trialstart==-1);
incorrect_end_times=find(N.binned_outcome.*N.binned_trialend==-1);
% coloring trajectory by state

t_start = behav(1,1) / 1000;
t_end   = behav(end,2) / 1000;

time_bins = t_start : bin_size : t_end;
t = time_bins(1:end-1) + bin_size/2;

% --- time vector for bins ---
t = time_bins(1:end-1) + bin_size/2;   % seconds

% --- map position → state ---
pos_time = pos(:,1) / 1000;

bin_idx = discretize(pos_time, time_bins);

state_per_pos = nan(size(pos_time));

valid = ~isnan(bin_idx) & bin_idx <= length(state_seq);

state_per_pos(valid) = state_seq(bin_idx(valid));

fig=figure
correct_trial_start = find(behav(:,4)==1);
ax1 = subplot(1,2,1);
hold(ax1,'on');
title(ax1,'Correct')
extra_start=0;
extra_end=3000;
for i = 1:3 % select correct trials

    start_time = behav(correct_trial_start(i),1);
    end_time   = behav(correct_trial_start(i),2);

    % --- indices ---
    trial_idx = find(pos(:,1) >= start_time & pos(:,1) <= end_time);
    trial_idx_extended = find(pos(:,1) >= start_time-extra_start & pos(:,1) <= end_time+extra_end);
    % --- plot trajectory colored by state ---
    for k = 1:length(trial_idx_extended)-1
        
        idx1 = trial_idx_extended(k);
        idx2 = trial_idx_extended(k+1);
        
        s = state_per_pos(idx1);  % state at this segment
        
        plot(pos([idx1 idx2],2), pos([idx1 idx2],3), ...
            'Color', colors(s,:), 'LineWidth', 2);
    end

    % --- markers ---
    [~, start_idx] = min(abs(pos(:,1) - start_time));
    plot(pos(start_idx,2), pos(start_idx,3), 'go', 'MarkerFaceColor',[0.5 0.5 0.5],'MarkerEdgeColor','none');
    plot(pos(trial_idx(end),2), pos(trial_idx(end),3), 'ro', 'MarkerFaceColor','k','MarkerEdgeColor','none');

end

xlim(ax1,[20,300])
ylim(ax1,[20,250])
xlabel(ax1,'x position')
ylabel(ax1,'y position')
hold(ax1,'off')

incorrect_trial_start = find(behav(:,4)==0);
ax2 = subplot(1,2,2);
hold(ax2,'on');
title(ax2,'Incorrect')

for i = 3:5 % select incorrect trials

    start_time = behav(incorrect_trial_start(i),1);
    end_time   = behav(incorrect_trial_start(i),2);

    % --- indices ---
    trial_idx = find(pos(:,1) >= start_time & pos(:,1) <= end_time);
    trial_idx_extended = find(pos(:,1) >= start_time-extra_start & pos(:,1) <= end_time+extra_end);
    
    % --- plot trajectory colored by state ---
    for k = 1:length(trial_idx_extended)-1
        
        idx1 = trial_idx_extended(k);
        idx2 = trial_idx_extended(k+1);
        
        s = state_per_pos(idx1);  % state at this segment
        
        plot(pos([idx1 idx2],2), pos([idx1 idx2],3), ...
            'Color', colors(s,:), 'LineWidth', 2);
    end

    % --- markers ---
    [~, start_idx] = min(abs(pos(:,1) - start_time));
    p(1)=plot(pos(start_idx,2), pos(start_idx,3), 'go', 'MarkerFaceColor',[0.5, 0.5, 0.5],'MarkerEdgeColor','none');
    p(2)=plot(pos(trial_idx(end),2), pos(trial_idx(end),3), 'ro', 'MarkerFaceColor','k','MarkerEdgeColor','none');

end


xlim(ax2,[20,300])
ylim(ax2,[20,250])
xlabel(ax2,'x position')
ylabel(ax2,'y position')
hold(ax2,'off')

lgd=legend(p, ["Trial start","Trial end"],'Location','southwest');
lgd.Position = [0.47, 0.9, 0.05, 0.07];  % Manual position outside

colormap(colors(1:numStates,:))
caxis([0.5 numStates + 0.5])
cb = colorbar(ax2);
cb.Position(1) = cb.Position(1) + 0.1; % push right
cb.Ticks = linspace(1,numStates,numStates);
cb.TickLabels = 1:numStates;
cb.Label.String = 'State';
```

<img src="Figures/state_trial.png" alt="Example trials" width="900">

```matlab
%% Compute and plot kernels

offset=1;
var_indices = {
    1:5;    % Trial start: 5 basis functions
    6:10;   % Trial start outcome: 5 basis functions
    11:15;   % Pre-trial end: 5 basis functions
    16:20;   % Pre-trial end outcome: 5 basis functions
    21:25;   % Trial end: 5 basis functions
    26:30;   % Trial end outcome: 5 basis functions
    31:35;   % Spike history: 5 basis function
    
};

kernels=struct();
for neuron_id=1:length(results.What)
    
    for g=1:length(var_indices)
        idx_group=var_indices{g}+offset;
        if  length(idx_group)>1
            weights = results.What{1,neuron_id}(idx_group,:);
            kernels(neuron_id).predictors{g} = bases.B(:,rcSelect) * weights;
            
        elseif length(idx_group)==1
            kernels(neuron_id).predictors{g}=results.What{1,neuron_id}(idx_group,:);
            
        end
    end
end
state_rwd = 3;
state_norwd = 2;
predictor=2;

for n = 1:length(results.What)
    kernel_rwd = kernels(n).predictors{1,predictor}(:,state_rwd);
    kernel_norwd = kernels(n).predictors{1,predictor}(:,state_norwd);
    
    if ~isempty(kernel_rwd)
        all_kernel_rwd(n,:) = kernel_rwd(:)'; % neurons x time
    end
    if ~isempty(kernel_norwd)
        all_kernel_norwd(n,:) = kernel_norwd(:)'; % neurons x time
    end
end
kernel_mean_rwd = nanmean(all_kernel_rwd, 1);
kernel_sd_rwd   = nanstd(all_kernel_rwd, 0, 1);
kernel_sem_rwd= kernel_sd_rwd ./ sqrt(sum(~isnan(all_kernel_rwd(:,1))));
kernel_mean_norwd = nanmean(all_kernel_norwd, 1);
kernel_sd_norwd   = nanstd(all_kernel_norwd, 0, 1);
kernel_sem_norwd = kernel_sd_norwd ./ sqrt(sum(~isnan(all_kernel_norwd(:,1))));
time_axis=(1:size(all_kernel_rwd,2))*bin_size;

fig = figure; hold on;

%Shaded region
fill([time_axis fliplr(time_axis)], ...
     [kernel_mean_rwd+kernel_sem_rwd fliplr(kernel_mean_rwd-kernel_sem_rwd)], ...
     colors(state_rwd,:), ...
     'FaceAlpha', 0.25, ...
     'EdgeColor', 'none');
fill([time_axis fliplr(time_axis)], ...
     [kernel_mean_norwd+kernel_sem_norwd fliplr(kernel_mean_norwd-kernel_sem_norwd)], ...
     colors(state_norwd,:), ...
     'FaceAlpha', 0.25, ...
     'EdgeColor', 'none');

%Mean line
p(1)=plot(time_axis, kernel_mean_rwd, ...
     'Color', colors(state_rwd,:), ...
     'LineWidth', 2);
p(2)=plot(time_axis, kernel_mean_norwd, ...
     'Color', colors(state_norwd,:), ...
     'LineWidth', 2);

xlabel('Time relative to trial start (s)');
ylabel('Kernel');
xlim([0 2])
legend(p,'Correct','Incorrect')
box off;
```

<img src="Figures/kernels_mpfc.png" alt="mPFC kernels" width="500">